# Train the V2 hierarchical geometry modelV2 training workflow for the robot-program-conditioned patch geometry with localized synthetic boundary learning. The encoder models conditional normal patch distributions; hierarchical Mahalanobis references, the file-state aggregation, and the trajectory/confidence layers are commissioned after selection.Key features:- Direct parameters: configure training via `TrainingParams` (e.g. `params = TrainingParams(profile="full", batch_size=32)`) or `V2_PROFILE` / `V2_SEED` / `V2_DATA_ROOT` / `V2_CHECKPOINT_PATH` environment variables.- Manual paths: run from the repository root and set `SRC_DIR`, `V2_DATA_ROOT`, and `V2_CHECKPOINT_PATH` in the first cells; the exact configured paths are used with fail-fast errors naming the variable to change.- Epoch profiles: `smoke=2` (contract), `balance=5` (coefficient balance), `full=50` (matched control/hybrid training) under one frozen configuration.- Hardware: CUDA when available (`torch.cuda.is_available()`), with transparent CPU fallback; one deterministic seed drives shuffling, corruption masks, and initialization.- Raw and weighted diagnostics: per-step `normal_loss` / `background_loss` / `boundary_loss` / ramped `alpha`, read-only gradient norms, and stationary clean-density selection on verified-healthy validation data.- Coherent resume and selection: `V2_RESUME=true` restores the exact coherent state; otherwise an existing checkpoint refuses silent overwrite and a missing resume checkpoint fails fast. Selection uses the stationary validation objective only, never test outcomes. There is no random-weight fallback anywhere.

In [ ]:
import json
import os
import sys
from dataclasses import dataclass
from pathlib import Path

import torch

os.environ.setdefault("PYTHONHASHSEED", "0")

# ---- Repository source (edit these one-line values; used exactly) ----
# Run notebooks from the repository root, or set V2_REPO_ROOT to the checkout path.
V2_REPO_ROOT = os.environ.get("V2_REPO_ROOT", ".")
SRC_DIR = os.environ.get("V2_SRC_DIR", str(Path(V2_REPO_ROOT) / "src"))

_source_dir = Path(SRC_DIR).expanduser()
if not (_source_dir / "representation").is_dir():
    raise FileNotFoundError(
        f"Repository source not found: SRC_DIR={_source_dir} has no 'representation' package. "
        "Run from the repository root or set V2_REPO_ROOT / V2_SRC_DIR to the checkout.")
_source_resolved = str(_source_dir.resolve())
if _source_resolved not in sys.path:
    sys.path.insert(0, _source_resolved)
print(f"[Env] Loaded representation modules from: {_source_dir}")

from representation.data import collate_variable_files
from representation.v2_aggregation import aggregate_file_state, calibrate_elevated_threshold
from representation.v2_config import V2Config
from representation.v2_inference import patch_regime_ids
from representation.v2_risk import HealthyTailCalibrator
from representation.v2_trainer import build_v2_training_stack
from representation.v2_trajectory import TrajectoryTracker
from synth.chronicle import load_chronological
from synth.config import PatchConfig
from synth.patchify import Patchifier
from synth.schema import SampleLabel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("[Hardware] Compute device:", device)
if device.type == "cuda":
    print("[Hardware] CUDA device name:", torch.cuda.get_device_name(0))

In [ ]:
# ---- Explicit data / checkpoint / output roots plus deterministic seed (edit these one-line values) ----
V2_DATA_ROOT = os.environ.get("V2_DATA_ROOT", "data/generated/chronicle-client")
V2_CHECKPOINT_PATH = os.environ.get("V2_CHECKPOINT_PATH", "checkpoints/v2_geometry.pt")
V2_OUTPUT_ROOT = os.environ.get("V2_OUTPUT_ROOT", "outputs/v2_training")
V2_PROFILE = os.environ.get("V2_PROFILE", "smoke")
V2_SEED = int(os.environ.get("V2_SEED", "0"))

@dataclass
class TrainingParams:
    """V2 training configuration (works from a repository checkout)."""
    data_root: str = V2_DATA_ROOT
    checkpoint_path: str = V2_CHECKPOINT_PATH
    output_root: str = V2_OUTPUT_ROOT
    profile: str = V2_PROFILE
    seed: int = V2_SEED
    batch_size: int = int(os.environ.get("V2_BATCH_SIZE", "8"))
    lr: float = float(os.environ.get("V2_LR", "1e-3"))
    corruption_rate: float = float(os.environ.get("V2_CORRUPTION_RATE", "0.1"))
    severity: float = float(os.environ.get("V2_SEVERITY", "2.0"))
    resume: bool = os.environ.get("V2_RESUME", "false").lower() in ("true", "1", "yes")
    d_model: int = int(os.environ.get("V2_D_MODEL", "32"))

params = TrainingParams()
EPOCH_PROFILES = {"smoke": 2, "balance": 5, "full": 50}
if params.profile not in EPOCH_PROFILES:
    raise ValueError(f"Unknown training profile {params.profile!r}; expected one of {sorted(EPOCH_PROFILES)}.")
if not 0.0 < params.corruption_rate < 1.0:
    raise ValueError("V2_CORRUPTION_RATE must be in (0, 1).")
total_epochs = EPOCH_PROFILES[params.profile]

configured_root = Path(params.data_root).expanduser()
DATA_ROOT = configured_root if configured_root.is_absolute() else Path.cwd() / configured_root
MANIFEST_PATH = DATA_ROOT / "manifest.json"
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        f"Chronological manifest not found at {MANIFEST_PATH}. Set V2_DATA_ROOT to the materialized "
        "chronicle root (generate it with notebooks/generate_chronological_factory.ipynb).")
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
print(f"[Config] profile={params.profile} epochs={total_epochs} seed={params.seed} device={device} "
      f"batch_size={params.batch_size} resume={params.resume}")

In [ ]:
# Load verified-healthy development views and build the V2 training stack.
torch.manual_seed(params.seed)

samples, manifest = load_chronological(DATA_ROOT)
by_id = {s.file_id: s for s in samples}
for view in ("dev_train", "dev_val"):
    if view not in manifest["splits"] or not manifest["splits"][view]:
        raise RuntimeError(f"Chronicle view {view!r} at {DATA_ROOT} is empty; regenerate the dataset.")
train_files = [by_id[i] for i in manifest["splits"]["dev_train"]]
val_files = [by_id[i] for i in manifest["splits"]["dev_val"]]
for sample in train_files + val_files:
    if sample.file_label is not SampleLabel.NORMAL:
        raise RuntimeError(f"V2 training fits verified-healthy files only; got {sample.file_id}.")
    if sample.split_provenance is not None and sample.split_provenance.is_quarantined:
        raise RuntimeError(f"Quarantined file {sample.file_id} must never train V2 geometry.")
n_robots = max(s.robot_idx for s in samples) + 1
n_programs = max(s.program_idx for s in samples) + 1
config = V2Config(n_channels=int(samples[0].x.shape[0]), d_model=params.d_model,
                  n_robots=n_robots, n_programs=n_programs, n_regimes=7, seed=params.seed)
patchifier = Patchifier(PatchConfig(patch_size=config.patch_size, stride=config.stride))
model, criterion, trainer = build_v2_training_stack(config, device=str(device), seed=params.seed, lr=params.lr)

def make_v2_batch(files, batch_patchifier, seed):
    """Collate full files plus seeded loss-only corruption masks (never encoder inputs)."""
    base = collate_variable_files(files, batch_patchifier)
    count = base["patches"].shape[1]
    regimes = patch_regime_ids(files, base["starts"], count)
    gen = torch.Generator().manual_seed(int(seed))
    corruption = (torch.rand(base["patch_valid_mask"].shape, generator=gen) < params.corruption_rate)
    corruption = corruption & base["patch_valid_mask"]
    return {"patches": base["patches"], "patch_pad_mask": base["patch_pad_mask"],
            "patch_valid_mask": base["patch_valid_mask"], "robot_idx": base["robot_idx"],
            "program_idx": base["program_idx"], "regime_ids": regimes,
            "corruption_mask": corruption, "severity": float(params.severity)}

def make_batches(files, batch_size, seed):
    return [make_v2_batch(files[start:start + batch_size], patchifier, seed + start)
            for start in range(0, len(files), batch_size)]

print(f"[Data] dev_train={len(train_files)} dev_val={len(val_files)} "
      f"n_robots={n_robots} n_programs={n_programs} d_model={config.d_model}")

In [ ]:
# Train over the selected epoch profile with coherent resume and stationary selection.
configured_ckpt = Path(params.checkpoint_path).expanduser()
checkpoint_path = configured_ckpt if configured_ckpt.is_absolute() else Path.cwd() / configured_ckpt
if params.resume:
    if not checkpoint_path.is_file():
        raise FileNotFoundError(
            f"V2 checkpoint not found at {checkpoint_path}. Train with V2_RESUME=false first "
            "or set V2_CHECKPOINT_PATH. Refusing to run on uninitialized weights.")
    resume_meta = trainer.load(checkpoint_path)
    print(f"[Resume] restored coherent state at step {resume_meta['step']} from {checkpoint_path}")
else:
    if checkpoint_path.is_file():
        raise FileExistsError(
            f"Checkpoint exists at {checkpoint_path}; set V2_RESUME=true to resume coherently "
            "or choose a fresh V2_CHECKPOINT_PATH. Refusing silent overwrite.")
    print(f"[Training] fresh {params.profile} run: {total_epochs} epochs from seed {params.seed}.")

history = []
val_probe = make_batches(val_files, params.batch_size, seed=params.seed + 1)
for epoch in range(1, total_epochs + 1):
    order_gen = torch.Generator().manual_seed(params.seed + epoch)
    order = torch.randperm(len(train_files), generator=order_gen).tolist()
    ordered = [train_files[i] for i in order]
    agg: dict[str, list[float]] = {}
    grad_norms: list[float] = []
    for start in range(0, len(ordered), params.batch_size):
        batch = make_v2_batch(ordered[start:start + params.batch_size], patchifier,
                              seed=params.seed + epoch * 1000 + start)
        terms = trainer.train_step(batch)
        for key, value in terms.items():
            agg.setdefault(key, []).append(value)
        total_sq = 0.0
        for param in model.parameters():
            if param.grad is not None:
                total_sq += float(param.grad.detach().pow(2).sum().item())
        grad_norms.append(total_sq ** 0.5)
    train_probe = make_batches(train_files, params.batch_size, seed=params.seed + 10000 + epoch)
    train_stat = trainer.stationary_loss(train_probe)
    val_stat = trainer.stationary_loss(val_probe)
    improved = trainer.record_eval(float(val_stat))
    summary = {key: sum(values) / len(values) for key, values in agg.items()}
    summary.update({"train_stationary": float(train_stat), "val_stationary": float(val_stat),
                    "grad_norm_mean": sum(grad_norms) / len(grad_norms),
                    "improved": improved, "step": trainer.step})
    history.append(summary)
    print(f"[Epoch {epoch}/{total_epochs}] loss={summary['loss']:.4f} normal={summary['normal_loss']:.4f} "
          f"background={summary['background_loss']:.4f} boundary={summary['boundary_loss']:.4f} "
          f"alpha={summary['alpha']:.3f} train_stat={train_stat:.4f} val_stat={val_stat:.4f} "
          f"grad_norm={summary['grad_norm_mean']:.3f} improved={improved}")
if trainer.best_state is None:
    raise RuntimeError("Stationary selection recorded no finite validation point; refusing to checkpoint.")
trainer.restore_best_state()
print(f"[Selection] restored best stationary={trainer.best_stationary:.4f} at step={trainer.step} "
      "(stationary validation objective only; test views untouched).")

In [ ]:
# Commission hierarchical references, trajectory baseline, and validation-only calibration.
model.eval()
latent_rows, robot_rows, program_rows, regime_rows, train_states = [], [], [], [], []
with torch.no_grad():
    for batch in make_batches(train_files, params.batch_size, seed=params.seed + 777):
        inputs = trainer._encoder_inputs(batch)
        encoded = model(**inputs)
        valid = batch["patch_valid_mask"]
        latent_rows.append(encoded["patch_latents"][valid])
        robot_rows.append(batch["robot_idx"].unsqueeze(1).expand_as(valid)[valid])
        program_rows.append(batch["program_idx"].unsqueeze(1).expand_as(valid)[valid])
        regime_rows.append(batch["regime_ids"][valid])
        state = aggregate_file_state(encoded["patch_latents"], encoded["patch_energy"], valid,
                                     batch["regime_ids"], top_q_fraction=config.top_q_fraction,
                                     elevated_threshold=config.elevated_threshold,
                                     n_regimes=config.n_regimes)
        train_states.append(state["file_state"])
geometry = trainer.fit_geometry(torch.cat(latent_rows), torch.cat(robot_rows),
                                torch.cat(program_rows), torch.cat(regime_rows),
                                torch.ones(sum(r.shape[0] for r in latent_rows), dtype=torch.bool))
train_states = torch.cat(train_states)
tracker = TrajectoryTracker(config.d_model)
tracker.fit_commissioning(train_states, torch.ones(train_states.shape[0], dtype=torch.bool))

# Validation-only aggregation calibration: elevated threshold from dev-val energies.
val_energy_parts, val_valid_parts, val_states = [], [], []
with torch.no_grad():
    for batch in make_batches(val_files, params.batch_size, seed=params.seed + 778):
        inputs = trainer._encoder_inputs(batch)
        encoded = model(**inputs)
        val_energy_parts.append(encoded["patch_energy"])
        val_valid_parts.append(batch["patch_valid_mask"])
        state = aggregate_file_state(encoded["patch_latents"], encoded["patch_energy"],
                                     batch["patch_valid_mask"], batch["regime_ids"],
                                     top_q_fraction=config.top_q_fraction,
                                     elevated_threshold=config.elevated_threshold,
                                     n_regimes=config.n_regimes)
        val_states.append(state["file_state"])
val_states = torch.cat(val_states)
width = max(part.shape[1] for part in val_energy_parts)
padded_energies = torch.cat([torch.cat([part, torch.zeros((part.shape[0], width - part.shape[1]))], dim=1)
                             for part in val_energy_parts], dim=0)
padded_valid = torch.cat([torch.cat([part, torch.zeros((part.shape[0], width - part.shape[1]),
                                                       dtype=torch.bool)], dim=1)
                          for part in val_valid_parts], dim=0)
elevation_floor = calibrate_elevated_threshold(padded_energies, padded_valid, tail_probability=0.05)
# Healthy-tail confidence needs enough validation files; tiny profiles pool the dev stream
# (server runs use dev-val only) while the floor itself stays validated.
if val_states.shape[0] >= config.calibration_min_samples:
    calibration_states, calibration_source = val_states, "dev-val only"
else:
    calibration_states = torch.cat([train_states, val_states])
    calibration_source = (f"pooled dev-train+dev-val ({val_states.shape[0]} < floor "
                          f"{config.calibration_min_samples}; server runs use dev-val only)")
probe_tracker = TrajectoryTracker(config.d_model)
probe_tracker.fit_commissioning(train_states, torch.ones(train_states.shape[0], dtype=torch.bool))
displacements = torch.cat([probe_tracker.update(row)["displacement"] for row in calibration_states])
calibrator = HealthyTailCalibrator(min_samples=config.calibration_min_samples).fit(displacements)

checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
trainer.save(checkpoint_path, geometry=geometry, calibrator=calibrator, tracker=tracker)
OUTPUT_DIR = Path(params.output_root).expanduser()
OUTPUT_DIR = OUTPUT_DIR if OUTPUT_DIR.is_absolute() else Path.cwd() / OUTPUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
run_summary = {"profile": params.profile, "epochs": total_epochs, "seed": params.seed,
               "data_root": str(DATA_ROOT), "checkpoint": str(checkpoint_path),
               "step": trainer.step, "best_stationary": trainer.best_stationary,
               "elevated_threshold": float(elevation_floor),
               "calibration_source": calibration_source, "history": history}
(OUTPUT_DIR / "v2-training-summary.json").write_text(json.dumps(run_summary, indent=2), encoding="utf-8")
print(f"[Checkpoint] saved coherent state at {checkpoint_path} step={trainer.step} "
      f"best_stationary={trainer.best_stationary:.4f} elevation={float(elevation_floor):.4f} "
      f"calibration={calibration_source}")